# Лабораторная работа: Колоризация

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import files
from PIL import Image
from pathlib import Path
import io,zipfile,tempfile,os
u=files.upload();name=next(n for n in u if n.endswith('.zip'))
root=tempfile.mkdtemp()
with zipfile.ZipFile(io.BytesIO(u[name])) as z:
 for member in z.infolist():
  dest=os.path.realpath(os.path.join(root,member.filename))
  assert dest.startswith(os.path.realpath(root)+os.sep) or dest==os.path.realpath(root)
 z.extractall(root)
paths=sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png'))[:1500]
rgb=[]
for p in paths:
 try:rgb.append(np.asarray(Image.open(p).convert('RGB').resize((128,128)),dtype='float32')/255)
 except Exception:pass
Y=np.stack(rgb);X=np.dot(Y,[.299,.587,.114]).astype('float32')[...,None]
from sklearn.model_selection import train_test_split
a,b,c,d=train_test_split(X,Y,test_size=.2,random_state=42)
inp=keras.Input((128,128,1));x=layers.Conv2D(32,3,padding='same',activation='relu')(inp)
x=layers.MaxPooling2D()(x);x=layers.Conv2D(64,3,padding='same',activation='relu')(x)
x=layers.MaxPooling2D()(x);x=layers.Conv2D(128,3,padding='same',activation='relu')(x)
x=layers.UpSampling2D()(x);x=layers.Conv2DTranspose(64,3,padding='same',activation='relu')(x)
x=layers.UpSampling2D()(x);model=keras.Model(inp,layers.Conv2DTranspose(3,3,padding='same',activation='sigmoid')(x))
model.compile('adam','mae',metrics=['mse'])
history=model.fit(a,c,validation_data=(b,d),epochs=20,batch_size=16)
pd.DataFrame(history.history).plot(grid=True);plt.show()
p=model.predict(b[:4]);fig,ax=plt.subplots(3,4,figsize=(12,7))
for i in range(4):
 ax[0,i].imshow(b[i,...,0],cmap='gray');ax[1,i].imshow(d[i]);ax[2,i].imshow(np.clip(p[i],0,1))
 for j in range(3):ax[j,i].axis('off')
plt.show()